# BirdCLEF+ 2026 — Baseline Submission

Strategy: EfficientNet-B0 (pretrained, frozen) → MLP head on quality-filtered balanced subset  
Metric: Macro ROC-AUC (skips classes with no true positive labels)  
Target runtime: ~20-25 min on Kaggle CPU (limit: 90 min)

In [6]:
import os, ast, time, warnings, multiprocessing
warnings.filterwarnings('ignore')
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import timm
from tqdm import tqdm

print(f'PyTorch  : {torch.__version__}')
print(f'CPU cores: {multiprocessing.cpu_count()}')

PyTorch  : 2.10.0
CPU cores: 10


## Config

In [ ]:
IS_KAGGLE   = Path('/kaggle').exists()
DATA_DIR    = Path('/kaggle/input/birdclef-2026') if IS_KAGGLE else Path('../data/raw')
WORKING_DIR = Path('/kaggle/working')             if IS_KAGGLE else Path('../outputs/submission')
MEL_CACHE   = WORKING_DIR / 'mels'
MODEL_PATH  = WORKING_DIR / 'best_model.pth'
BACKBONE_PATH = WORKING_DIR / 'backbone.pth'
SUB_PATH    = WORKING_DIR / 'submission.csv'
WORKING_DIR.mkdir(parents=True, exist_ok=True)
MEL_CACHE.mkdir(parents=True, exist_ok=True)

SR, DURATION        = 32000, 5.0
N_MELS, N_FFT       = 128, 1024
HOP_LENGTH          = 320
FMIN, FMAX          = 20.0, 16000.0
MAX_PER_CLASS       = 50      # more data per class
MIN_RATING          = 2.0     # filter out low-quality recordings
INF_STRIDE          = 2.5     # overlapping windows at inference (catches boundary birds)
EPOCHS              = 15
BATCH_SIZE          = 64
LR                  = 1e-3
LABEL_SMOOTH        = 0.05    # soften noisy secondary labels
MODEL_NAME          = 'efficientnet_b0'
SEED                = 42
torch.manual_seed(SEED); np.random.seed(SEED)

print(f'Data dir : {DATA_DIR}  exists={DATA_DIR.exists()}')

## 1. Load Metadata & Balance Dataset

In [ ]:
df = pd.read_csv(DATA_DIR / 'train.csv')
df['primary_label'] = df['primary_label'].astype(str)
df['secondary_labels'] = df['secondary_labels'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

species_list = sorted(df['primary_label'].unique().tolist())
NUM_CLASSES  = len(species_list)
s2i          = {s: i for i, s in enumerate(species_list)}
print(f'Total recordings: {len(df):,}  |  Species: {NUM_CLASSES}')

# Quality filter + balanced subset (prefer higher-rated recordings)
df_qual   = df[df['rating'] >= MIN_RATING].copy()
# For species with no recordings above threshold, fall back to all their recordings
rare = df.groupby('primary_label').filter(
    lambda g: (g['rating'] >= MIN_RATING).sum() == 0)['primary_label'].unique()
if len(rare):
    df_qual = pd.concat([df_qual, df[df['primary_label'].isin(rare)]], ignore_index=True)

df_sorted = df_qual.sort_values('rating', ascending=False)
df_bal = (df_sorted.groupby('primary_label', group_keys=False)
          .head(MAX_PER_CLASS)
          .reset_index(drop=True))
df_bal = df_bal.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'After quality filter (rating>={MIN_RATING}): {len(df_qual):,}')
print(f'Balanced subset: {len(df_bal):,}  |  Species: {df_bal["primary_label"].nunique()} / {NUM_CLASSES}')

# Stratified split — rare species (<2 samples) go entirely to train
counts     = df_bal['primary_label'].value_counts()
rare_mask  = df_bal['primary_label'].isin(counts[counts < 2].index)
train_only = df_bal[rare_mask]
splittable = df_bal[~rare_mask].reset_index(drop=True)

from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
trn_idx, val_idx = next(sss.split(splittable, splittable['primary_label']))
trn_df = pd.concat([train_only, splittable.iloc[trn_idx]], ignore_index=True)
val_df  = splittable.iloc[val_idx].reset_index(drop=True)
print(f'Train: {len(trn_df):,}  Val: {len(val_df):,}')
print(f'Val species coverage: {val_df["primary_label"].nunique()} / {NUM_CLASSES}')

## 2. Pre-compute Mels (balanced subset only)

In [9]:
def load_and_mel(filename, audio_dir, cache_dir):
    cache_path = cache_dir / (filename.replace('/', '__').replace('.ogg', '.npy'))
    if cache_path.exists():
        return True
    try:
        wav, native_sr = sf.read(str(audio_dir / filename), dtype='float32')
        if wav.ndim > 1: wav = wav[:, 0]
        if native_sr != SR:
            wav = librosa.resample(wav, orig_sr=native_sr, target_sr=SR)
        target_len = int(SR * DURATION)
        if len(wav) <= target_len:
            wav = np.pad(wav, (0, target_len - len(wav)))
        else:
            start = np.random.randint(0, len(wav) - target_len)
            wav = wav[start: start + target_len]
        mel = librosa.feature.melspectrogram(y=wav, sr=SR, n_mels=N_MELS,
            n_fft=N_FFT, hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
        np.save(cache_path, mel_db.astype(np.float32))
        return True
    except Exception:
        return False

t0 = time.time()
audio_dir = DATA_DIR / 'train_audio'
already   = sum(1 for _ in MEL_CACHE.glob('*.npy'))
print(f'Already cached: {already} / {len(df_bal)}')

for _, row in tqdm(df_bal.iterrows(), total=len(df_bal), desc='Mel cache'):
    load_and_mel(row['filename'], audio_dir, MEL_CACHE)

print(f'Done in {time.time()-t0:.0f}s  |  Cached: {sum(1 for _ in MEL_CACHE.glob("*.npy"))}')

Already cached: 3652 / 3652


Mel cache: 100%|██████████| 3652/3652 [00:00<00:00, 12451.50it/s]

Done in 0s  |  Cached: 3652


## 3. Extract Backbone Features (once, no grad)

In [ ]:
# Load backbone — try pretrained, fallback to cached weights
backbone = timm.create_model(MODEL_NAME, pretrained=False, in_chans=1, num_classes=0)
FEAT_DIM = backbone.num_features

if BACKBONE_PATH.exists():
    backbone.load_state_dict(torch.load(BACKBONE_PATH, map_location='cpu'))
    print(f'Backbone loaded from cache: {BACKBONE_PATH}')
else:
    try:
        backbone = timm.create_model(MODEL_NAME, pretrained=True, in_chans=1, num_classes=0)
        torch.save(backbone.state_dict(), BACKBONE_PATH)
        print(f'Backbone downloaded + cached to {BACKBONE_PATH}')
    except Exception as e:
        print(f'Pretrained download failed ({e}), using random weights.')

backbone.eval()
print(f'Backbone: {MODEL_NAME}  |  Feature dim: {FEAT_DIM}')

def extract_features(df_split, smooth=True):
    feats, labels_list = [], []
    for i in tqdm(range(0, len(df_split), BATCH_SIZE), desc='Extract feats'):
        batch_rows = df_split.iloc[i: i + BATCH_SIZE]
        mels = []
        for _, row in batch_rows.iterrows():
            p = MEL_CACHE / (row['filename'].replace('/', '__').replace('.ogg', '.npy'))
            mels.append(np.load(p))
        mel_t = torch.from_numpy(np.stack(mels)).unsqueeze(1)
        with torch.no_grad():
            feats.append(backbone(mel_t).numpy())
        # Labels: primary=1.0, secondary=1-LABEL_SMOOTH (softer, less noisy)
        lbl = np.zeros((len(batch_rows), NUM_CLASSES), dtype=np.float32)
        for j, (_, row) in enumerate(batch_rows.iterrows()):
            if row['primary_label'] in s2i:
                lbl[j, s2i[row['primary_label']]] = 1.0
            for s in row['secondary_labels']:
                if s in s2i:
                    lbl[j, s2i[s]] = max(lbl[j, s2i[s]], 1.0 - LABEL_SMOOTH if smooth else 1.0)
        labels_list.append(lbl)
    return np.concatenate(feats), np.concatenate(labels_list)

t0 = time.time()
trn_feats, trn_labels = extract_features(trn_df, smooth=True)
val_feats, val_labels = extract_features(val_df, smooth=False)  # binary for AUC
print(f'Features extracted in {time.time()-t0:.0f}s')
print(f'Train features: {trn_feats.shape}  Val: {val_feats.shape}')

## 4. Train Linear Head

In [ ]:
from torch.utils.data import TensorDataset

def macro_roc_auc(targets, preds):
    scores = [roc_auc_score(targets[:, i], preds[:, i])
              for i in range(targets.shape[1]) if targets[:, i].sum() > 0]
    return float(np.mean(scores)) if scores else 0.0

trn_ds = TensorDataset(torch.from_numpy(trn_feats), torch.from_numpy(trn_labels))
val_ds = TensorDataset(torch.from_numpy(val_feats), torch.from_numpy(val_labels))
trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader  = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# MLP head: 1280 -> 512 -> 206 (stronger than single linear layer)
head = nn.Sequential(
    nn.Linear(FEAT_DIM, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, NUM_CLASSES),
)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_auc = 0.0
for epoch in range(1, EPOCHS + 1):
    head.train()
    trn_loss = 0.0
    for feats_b, lbl_b in trn_loader:
        optimizer.zero_grad()
        loss = criterion(head(feats_b), lbl_b)
        loss.backward(); optimizer.step()
        trn_loss += loss.item()
    trn_loss /= len(trn_loader)

    head.eval()
    all_preds, all_tgts = [], []
    with torch.no_grad():
        for feats_b, lbl_b in val_loader:
            all_preds.append(torch.sigmoid(head(feats_b)).numpy())
            all_tgts.append(lbl_b.numpy())
    auc = macro_roc_auc(np.concatenate(all_tgts), np.concatenate(all_preds))
    scheduler.step()

    print(f'Ep {epoch:02d} | train_loss={trn_loss:.4f} | val_auc={auc:.4f}')
    if auc > best_auc:
        best_auc = auc
        torch.save({'backbone': MODEL_NAME, 'head': head.state_dict(),
                    'species_list': species_list}, MODEL_PATH)

print(f'Best val AUC: {best_auc:.4f}')

## 5. Inference on Test Soundscapes

In [ ]:
ckpt = torch.load(MODEL_PATH, map_location='cpu')
head.load_state_dict(ckpt['head'])
backbone.eval(); head.eval()

test_dir   = DATA_DIR / 'test_soundscapes'
test_files = sorted(test_dir.glob('*.ogg'))
print(f'Test soundscapes: {len(test_files)}')

def mel_from_wave(wav):
    mel = librosa.feature.melspectrogram(y=wav, sr=SR, n_mels=N_MELS,
        n_fft=N_FFT, hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    return mel_db.astype(np.float32)

def predict_soundscape(audio_path):
    wav, native_sr = sf.read(str(audio_path), dtype='float32')
    if wav.ndim > 1: wav = wav[:, 0]
    if native_sr != SR:
        wav = librosa.resample(wav, orig_sr=native_sr, target_sr=SR)

    target_len  = int(SR * DURATION)
    stride_len  = int(SR * INF_STRIDE)   # overlapping windows
    windows, end_secs = [], []
    for start in range(0, len(wav) - target_len + 1, stride_len):
        windows.append(wav[start: start + target_len])
        # end_sec rounded to nearest 5s to match submission row_id format
        end_s = int(round((start + target_len) / SR / 5) * 5)
        end_secs.append(max(end_s, 5))
    if not windows:
        windows.append(np.pad(wav, (0, target_len - len(wav))))
        end_secs.append(5)

    # Accumulate max prob per row_id (handles duplicate end_secs from overlap)
    row_preds = {}
    stem = audio_path.stem
    for i in range(0, len(windows), BATCH_SIZE):
        batch_wav = windows[i: i + BATCH_SIZE]
        mels = torch.from_numpy(np.stack([mel_from_wave(w) for w in batch_wav])).unsqueeze(1)
        with torch.no_grad():
            probs = torch.sigmoid(head(backbone(mels))).numpy()
        for probs_row, end_s in zip(probs, end_secs[i: i + BATCH_SIZE]):
            rid = f'{stem}_{end_s}'
            if rid not in row_preds:
                row_preds[rid] = probs_row
            else:
                row_preds[rid] = np.maximum(row_preds[rid], probs_row)

    rows = []
    for rid, probs_row in row_preds.items():
        row = {'row_id': rid}
        for j, sp in enumerate(species_list):
            row[sp] = float(probs_row[j])
        rows.append(row)
    return rows

all_rows = []
for fp in tqdm(test_files, desc='Inference'):
    all_rows.extend(predict_soundscape(fp))

if all_rows:
    submission = pd.DataFrame(all_rows)
else:
    # No test soundscapes found — use sample_submission as template with uniform scores
    print('WARNING: No test soundscapes found. Using sample_submission as fallback.')
    submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
    for sp in species_list:
        if sp in submission.columns:
            submission[sp] = 1.0 / NUM_CLASSES  # uniform baseline

submission.to_csv(SUB_PATH, index=False)
print(f'Saved {len(submission):,} rows  {submission.shape[1]} cols  to {SUB_PATH}')
submission.head(3)

## 6. Validate Submission

In [ ]:
sample_sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
missing_cols = set(sample_sub.columns) - set(submission.columns)
print(f'Shape   : {submission.shape}')
print(f'Missing : {missing_cols if missing_cols else "none"}')

if len(submission) > 0:
    in_range = ((submission[species_list] >= 0) & (submission[species_list] <= 1)).all().all()
    print(f'Scores in [0,1]: {in_range}')
else:
    print('No test soundscapes found (expected on Kaggle rerun only).')

print(f'Saved to: {SUB_PATH}')

## 7. Submit to Competition

In [ ]:
# On Kaggle: submission.csv at /kaggle/working/submission.csv is auto-submitted on commit.
# Locally: use kaggle CLI to submit directly.

print(f'Submission file : {SUB_PATH}')
print(f'File size       : {SUB_PATH.stat().st_size / 1024:.1f} KB' if SUB_PATH.exists() else 'File not found!')
print(f'Rows            : {len(submission):,}')

if IS_KAGGLE:
    print()
    print('Running on Kaggle — submission.csv will be auto-submitted when notebook is committed.')
    print('No action needed.')
else:
    # Local submission via Kaggle API
    import subprocess, sys
    msg = f'EfficientNet-B0 frozen + MLP head | val_auc={best_auc:.4f} | {len(df_bal)} samples'
    print(f'\nSubmitting locally: {msg}')
    result = subprocess.run(
        ['kaggle', 'competitions', 'submit',
         '-c', 'birdclef-2026',
         '-f', str(SUB_PATH),
         '-m', msg],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f'Error: {result.stderr}')
        print('Note: Local submission only works if submission.csv is non-empty (needs test soundscapes).')